<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/HW/01RAD_Ex04_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 04 (team work): The linear algebra of house prices

The file `sarasota_houses_mod.csv` contains 1057 houses sold in one region of the USA with the sale price and a few characteristics:

| column | description |
| --- | --- |
| `price` | sale price in USD |
| `living_area` | interior living area in square feet |
| `bathrooms` | number of bathrooms (can be fractional) |
| `bedrooms` | number of bedrooms |
| `fireplaces` | number of fireplaces |
| `lot_size` | lot size in acres |
| `age` | age of the house in years |
| `fireplace` | indicator whether the house has at least one fireplace |

The homework applies Lecture 4 and Exercise 04 to a real data set: the model matrix, the normal equations as orthogonality, the projection matrices $\mathbf{H}$ and $\mathbf{M}$, reparametrisation, the Frisch–Waugh–Lovell theorem, rank deficiency, and the numerics of least squares (QR, SVD, condition number). Tasks 09 and 10 use Lecture 5 and Exercise 05: the distribution of $\widehat{\boldsymbol{\beta}}$ (Theorems 3.4 and 3.5), dummy variables and interactions. Most tasks can be checked against `statsmodels`; the point is to understand *why* the numbers come out as they do. Write a short answer in a markdown cell after each task.

## Team work
This homework is larger than the previous ones and is solved **in teams of two or three students**. Split the tasks as you like, but every member should be able to explain every answer. At the top of the notebook list the team members and, in one line each, who did what.

## Submission
**Deadline: Sunday 1 November 2026, 23:59** (two weeks, after Exercise 05). Submit through GitHub as before: fork `francji1/01RAD`, save **one notebook per team** into the `HW/` folder as `01RAD_Ex04_HW_<Surname1>_<Surname2>.ipynb` (surnames of all members joined by an underscore, without the brackets, for example `01RAD_Ex04_HW_Novak_Svoboda.ipynb`) and open one pull request to `main`. The pull request should add your one file only; do not modify or delete other files. Python or R, as you prefer.

Useful calls: `patsy.dmatrices`, `np.linalg.qr`, `np.linalg.svd`, `np.linalg.matrix_rank`, `np.linalg.cond`, `scipy.linalg.solve_triangular`, `sm.OLS(y, X).fit()` and `.fit(method='qr')`, `model.df_model`, `model.scale`, `model.get_influence().hat_matrix_diag`.

## Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.linalg as sla
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices

sns.set_theme(style='whitegrid', palette='deep')

url_houses = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/sarasota_houses_mod.csv'
houses = pd.read_csv(url_houses)
print(f'Rows: {houses.shape[0]}, columns: {houses.shape[1]}')
houses.describe().T

## Task 01 - Data and the model matrix
Check the missing values and remove the houses with a missing `price` (the column `lot_size` is not used in this homework, so its missing values do not matter). For the model

`price ~ living_area + bathrooms + bedrooms + age + fireplace`

build the model matrix $\mathbf{X}$ once by hand with numpy and once with `dmatrices`, and check that they agree. State $n$, $m+1$, the rank $h(\mathbf{X})$, and what the column for `fireplace` contains.

In [ ]:
# your solution


## Task 02 - Your own least squares
Write a function `my_ols(X, y)` that uses **only** the QR decomposition (`np.linalg.qr` and a triangular solver, no inverse, no `lstsq`) and returns $\widehat{\boldsymbol{\beta}}$, $\widehat{\mathbf{y}}$, $\widehat{\mathbf{e}}$, $s_n^2$ and the leverages $h_{ii}$. Hint: $\mathbf{H} = \mathbf{Q}\mathbf{Q}^T$, so $h_{ii}$ is the squared norm of the $i$-th row of $\mathbf{Q}$; do not build the $n \times n$ matrix.

Apply it to the model of Task 01 and compare $\widehat{\boldsymbol{\beta}}$, $s_n^2$ (`model.scale`), $R^2$ and $h_{ii}$ (`hat_matrix_diag`) with `statsmodels`. Verify numerically $\mathbf{X}^T\widehat{\mathbf{e}} = \mathbf{0}$, $\sum_i h_{ii} = m+1$, $\|\mathbf{y}\|^2 = \|\widehat{\mathbf{y}}\|^2 + \|\widehat{\mathbf{e}}\|^2$ and $R^2 = \mathrm{corr}^2(\mathbf{y}, \widehat{\mathbf{y}})$.

In [ ]:
# your solution


## Task 03 - Units and reparametrisation
We now want the price in **thousands of USD**, the living area in **square metres** ($1\ \mathrm{ft}^2 = 0.09290304\ \mathrm{m}^2$) and the age **centred** at its mean.

1. Show that the new model matrix is $\widetilde{\mathbf{X}} = \mathbf{X}\mathbf{A}$ for a regular matrix $\mathbf{A}$ and write $\mathbf{A}$ explicitly. With $\widetilde{\mathbf{y}} = c\,\mathbf{y}$, derive $\widetilde{\boldsymbol{\beta}} = c\,\mathbf{A}^{-1}\widehat{\boldsymbol{\beta}}$ from the normal equations.
2. **Before fitting**, compute the new coefficients from the old ones with this formula. Then fit the new model and compare.
3. Which of the following change and how: $\widehat{\mathbf{y}}$, $\widehat{\mathbf{e}}$, $s_n$, $R^2$, $h_{ii}$, $\kappa(\mathbf{X})$? Explain with $\mathbf{H}$.
4. Interpret the intercept of the new model in one sentence. Why is the intercept of the original model hard to interpret?

In [ ]:
# your solution


## Task 04 - The bedroom paradox
Fit the simple regression `price ~ bedrooms` and compare its slope with the coefficient of `bedrooms` in the model of Task 01. The signs differ.

1. Verify the Frisch–Waugh–Lovell theorem for `bedrooms`: compute $\mathbf{M}_{-j}\mathbf{x}_j$ and $\mathbf{M}_{-j}\mathbf{y}$ (residuals of the regressions of `bedrooms` and of `price` on the other columns) and show that the slope of one on the other equals the multiple regression coefficient.
2. Draw both scatter plots (raw and cleaned) with their lines.
3. How much of the variability of `bedrooms` is explained by the other regressors ($R^2$ of the auxiliary regression)?
4. Explain the paradox in plain words for a real estate agent: what does the negative coefficient say about two houses that differ in the number of bedrooms but **not** in the other variables?

In [ ]:
# your solution


## Task 05 - Dummy variables and the trap
1. Add the column `no_fireplace = 1 - fireplace` to the model matrix of Task 01 (keeping the intercept). What is the rank? Fit the model with `sm.OLS(y, X).fit()` and with your `my_ols`. Which coefficients are identified (i.e. are the same for every solution of the normal equations)? Express them through the output. What would R's `lm` report?
2. Now remove the intercept and keep **both** dummies. What is the rank now? Show that this model has exactly the same fitted values as the model of Task 01 and explain why with the column space $\mathcal{M}(\mathbf{X})$. How are the coefficients of the two parametrisations related?
3. Compare the `R-squared` reported by `statsmodels` for the model in 2. with Task 01. In Exercise 03 the model through the origin reported the uncentred $R^2 = \sum_i \widehat{y}_i^2 / \sum_i y_i^2$. Is that the case here? Explain why not (hint: is the vector $\mathbf{1}$ in $\mathcal{M}(\mathbf{X})$?) and what really decides whether a model "has an intercept".

In [ ]:
# your solution


## Task 06 - Leverage as a data check
Plot the leverages $h_{ii}$ of the model of Task 01 against the observation index and list the ten houses with the largest $h_{ii}$. Compare with the mean leverage $(m+1)/n$.

1. What do the houses with the largest leverage have in common? Look at the variable `age` in particular: how many houses are older than 100 and older than 200 years, and how do they differ from the rest in the other variables? Are such values plausible, and what would you check or ask before using them?
2. Refit the model without the houses with `age` above 150 years and compare the coefficients, in particular the coefficient of `age`. Does the decision matter?
3. Leverage depends only on $\mathbf{X}$. Why is it nevertheless a good tool for finding errors in the data, before the response is even used?

In [ ]:
# your solution


## Task 07 - When the default fails: polynomials in the living area
Fit polynomial models `price` $= \beta_0 + \beta_1 a + \beta_2 a^2 + \dots + \beta_d a^d + e$ in the living area $a$ (square feet, raw) for degrees $d = 1, \dots, 6$, building the columns $a^k$ yourself (`np.vander`). For each degree record in one table:
- $\kappa(\mathbf{X})$ and `np.linalg.matrix_rank(X)`,
- $R^2$ and `df_model` from `sm.OLS(y, X).fit()` (default, pseudoinverse),
- $R^2$ from `sm.OLS(y, X).fit(method='qr')` and from your `my_ols`,
- $R^2$ with the standardized variable $(a - \overline{a})/s_a$ instead of $a$.

1. The models are **nested**: the model of degree $d+1$ contains the model of degree $d$. What does that imply for $R^2$ as $d$ grows (think of the projection onto a larger subspace)? Which column of the table violates this, and from which degree on?
2. Explain what the default method does wrong. Hint: compute the singular values of $\mathbf{X}$, compare them with the tolerance used to decide which ones count as zero (`np.linalg.matrix_rank` uses $s_1 \cdot \max(n, p) \cdot \varepsilon_{\text{machine}}$), and look at `df_model`.
3. Why do QR and the standardized version give the correct answer? Which fix would you use in practice?

In [ ]:
# your solution


## Task 08 - Singular values and the variance of the estimates
Take the model of Task 01 with all non-intercept columns standardized (including `fireplace`), so that the coefficients are comparable.

1. Compute the SVD $\mathbf{X} = \mathbf{P}\mathbf{S}\mathbf{Q}^T$, the singular values and $\kappa(\mathbf{X})$. Is there a problem with collinearity in this model?
2. `statsmodels` reports the standard errors $\sqrt{s_n^2\,[(\mathbf{X}^T\mathbf{X})^{-1}]_{jj}}$ (Theorem 3.4 will justify this in the next lecture). Using the SVD, write $[(\mathbf{X}^T\mathbf{X})^{-1}]_{jj} = \sum_k q_{jk}^2/s_k^2$ and compute the standard errors from this formula; compare with `model.bse`.
3. For the coefficient of `bedrooms`, compute the share of each term $q_{jk}^2/s_k^2$ in the sum. Which singular direction contributes most? Which variables have large weights in that direction $\mathbf{q}_k$, and how does it relate to Task 04?

In [ ]:
# your solution


## Task 09 - Theorem 3.4 by simulation
Take the model of Task 01 as the truth: the model matrix $\mathbf{X}$ is fixed (the 1055 houses), the true parameters are $\boldsymbol{\beta} = \widehat{\boldsymbol{\beta}}$ and $\sigma = s_n$ from Task 01. Generate at least 2000 new response vectors $\mathbf{y}^* = \mathbf{X}\boldsymbol{\beta} + \mathbf{e}^*$, $\mathbf{e}^* \sim N_n(\mathbf{0}, \sigma^2\mathbf{I}_n)$, and in each replication compute $\widehat{\boldsymbol{\beta}}^*$ and $s_n^{2*}$ (use your `my_ols`).

1. Compare the mean of $\widehat{\boldsymbol{\beta}}^*$ with $\boldsymbol{\beta}$ and the mean of $s_n^{2*}$ with $\sigma^2$ (Theorem 3.4, points 1 and 3).
2. Compare the empirical covariance matrix of $\widehat{\boldsymbol{\beta}}^*$ with $\sigma^2(\mathbf{X}^T\mathbf{X})^{-1}$: the standard deviations of the coefficients and the correlation matrix.
3. The estimates of the coefficients of `bedrooms` and `living_area` are correlated. Which sign does the correlation have, and why? Draw the scatter plot of the two estimates and relate it to the correlation of the regressors and to Task 04.
4. For `bedrooms`, check that $T = (\widehat{\beta}_j^* - \beta_j)/\big(s_n^*\sqrt{[(\mathbf{X}^T\mathbf{X})^{-1}]_{jj}}\big)$ has the $t(n-m-1)$ distribution (Theorem 3.5): histogram with the density, and the share of replications in which the 95 % confidence interval covers the true $\beta_j$.

In [ ]:
# your solution


## Task 10 - Fireplace: interaction and separate regressions
Fit the model `price ~ living_area * fireplace` (R: `lm(price ~ living_area * fireplace)`).

1. Write the model matrix for this formula: which four columns does it have? Write the two regression lines (houses without and with a fireplace) in terms of the four coefficients and interpret each coefficient in one sentence, including its units.
2. Test whether the price per square foot is the same for houses with and without a fireplace (the $t$-test of the interaction term). State the hypotheses, the statistic, the degrees of freedom and the conclusion. Draw the data with the two fitted lines.
3. Now fit two separate simple regressions `price ~ living_area`, one for the houses with and one for the houses without a fireplace. Compare the coefficients with the lines from 1.: what is equal and why? (Think of the column space of the interaction model and of a model matrix that is block diagonal after reordering the rows.)
4. Compare $s_n$ and the standard errors of the slopes in the interaction model and in the two separate models. Which assumption about $\sigma^2$ makes the difference? Which approach would you prefer here, and what would you check to decide?

In [ ]:
# your solution


## Task 11 - Team summary
In 6 to 10 sentences, as a team: what did the linear-algebra view explain in these data that the `summary()` table alone would not have shown? Which numerical method would you trust for least squares and why? Name one thing in the data you would report to whoever prepared them.

In [ ]:
# your solution
